<a href="https://colab.research.google.com/github/ghadirchhade/Master-Thesis/blob/main/E05_w_tiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!pip install -q torch torchvision

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# facebook/sam3 is gated on the Hub -> you must accept the license at
# https://huggingface.co/facebook/sam3 with the account whose token you use below.
!pip install -q -U transformers accelerate huggingface_hub supervision

from huggingface_hub import login
login()  # paste your HF token (needs access to facebook/sam3)

print("Transformers SAM3 dependencies installed.")

In [ ]:
import os
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import supervision as sv
import matplotlib.patches as patches
from transformers import Sam3Model, Sam3Processor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

sam3_model = Sam3Model.from_pretrained("facebook/sam3", device_map="auto")
sam3_model.eval()

sam3_processor = Sam3Processor.from_pretrained("facebook/sam3")

print("HF transformers SAM3 model + processor loaded.")
print("Model device:", next(sam3_model.parameters()).device)

In [ ]:
!pip install -q jupyter_bbox_widget

In [ ]:
#  Load image, GT boxes, visualize indexed GT boxes
IMAGE_PATH = "/content/drive/MyDrive/datasets/AGS_Multi_Rumex/images/20230426_Wallenwil/DJI_20230426110736_0035.JPG"
LABEL_PATH = "/content/drive/MyDrive/datasets/AGS_Multi_Rumex/annotations_yolo/DJI_20230426110736_0035.txt"
RUMEX_CLASS_ID = 0

target_image = Image.open(IMAGE_PATH).convert("RGB")
img_w, img_h = target_image.size
print(f"Target image size (W, H): {target_image.size}")

def load_yolo_boxes(label_path, img_width, img_height, class_id=0):
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            if cls != class_id:
                continue
            xc, yc, bw, bh = map(float, parts[1:5])
            xc, yc, bw, bh = xc * img_width, yc * img_height, bw * img_width, bh * img_height
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
    return np.array(boxes, dtype=np.float32)

gt_boxes = load_yolo_boxes(LABEL_PATH, img_w, img_h, class_id=RUMEX_CLASS_ID)
print(f"Loaded {len(gt_boxes)} GT box(es) with class id={RUMEX_CLASS_ID}.")

import matplotlib.font_manager as fm
font_path = fm.findfont("DejaVu Sans")
font = ImageFont.truetype(font_path, 100)

gt_preview = target_image.copy()
draw = ImageDraw.Draw(gt_preview)
for i, (x0, y0, x1, y1) in enumerate(gt_boxes):
    draw.rectangle([x0, y0, x1, y1], outline=(255, 255, 0), width=20)
    text_x, text_y = int(x0), max(0, int(y0) - 120)
    draw.rectangle([text_x, text_y, text_x + 120, text_y + 120], fill=(255, 255, 255))
    draw.text((text_x + 10, text_y + 5), str(i), fill=(255, 0, 0), font=font)

plt.figure(figsize=(10, 8))
plt.imshow(gt_preview)
plt.title("All GT boxes (yellow), indexed")
plt.axis("off")
plt.show()

In [ ]:
#  Select POSITIVE exemplars by GT index, crop them
positive_indices = [2, 6, 9]   # <-- edit: indices into gt_boxes to use as positive exemplars

pos_boxes_fullres = [
    [float(x0), float(y0), float(x1), float(y1)]
    for (x0, y0, x1, y1) in gt_boxes[positive_indices]
]
pos_crops = [
    target_image.crop([int(round(v)) for v in box]) for box in pos_boxes_fullres
]

print(f"Positive exemplar indices: {positive_indices}")
for i, box in zip(positive_indices, pos_boxes_fullres):
    print(f"  idx {i}: box={[round(v, 1) for v in box]}")

fig, axes = plt.subplots(1, len(pos_crops), figsize=(4 * len(pos_crops), 4))
if len(pos_crops) == 1:
    axes = [axes]
for ax, crop, idx in zip(axes, pos_crops, positive_indices):
    ax.imshow(crop)
    ax.set_title(f"POS idx {idx}")
    ax.axis("off")
plt.show()

In [ ]:
# Interactive bounding box widget (NEGATIVE boxes only)
# Draw boxes around regions that are NOT rumex -- grass, soil, clover,
# another weed species, shadows, mulch, etc. These become negative
# exemplars, combined with the positive exemplars to help suppress
# false-positive matches on lookalikes.

from jupyter_bbox_widget import BBoxWidget
import io, base64

def pil_to_base64(img):
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()

WIDGET_MAX_DIM = 1400

def resize_for_widget(img, max_dim=WIDGET_MAX_DIM):
    w, h = img.size
    scale = max_dim / max(w, h)
    if scale >= 1:
        return img, 1.0
    new_w, new_h = int(w * scale), int(h * scale)
    return img.resize((new_w, new_h), Image.BILINEAR), scale

image_widget_display, widget_scale = resize_for_widget(target_image)

neg_widget = BBoxWidget(
    image=pil_to_base64(image_widget_display),
    classes=["negative"],
)

print("Draw boxes around NON-rumex regions (grass, soil, other plants, etc.)")
print(f"Displayed at scale={widget_scale:.4f} of full-res -- coordinates will be rescaled after.")
neg_widget

In [ ]:
# Extract negative boxes from widget, crop them, sanity-check
negative_boxes_fullres = []
for box in neg_widget.bboxes:
    x0 = box["x"] / widget_scale
    y0 = box["y"] / widget_scale
    x1 = (box["x"] + box["width"]) / widget_scale
    y1 = (box["y"] + box["height"]) / widget_scale
    negative_boxes_fullres.append([float(x0), float(y0), float(x1), float(y1)])

print(f"Collected {len(negative_boxes_fullres)} negative box(es) (full-res coords):")
for b in negative_boxes_fullres:
    print(" ", [round(v, 1) for v in b])

neg_crops = [
    target_image.crop([int(round(v)) for v in box]) for box in negative_boxes_fullres
]

check_preview = target_image.copy()
draw = ImageDraw.Draw(check_preview)
for (nx0, ny0, nx1, ny1) in negative_boxes_fullres:
    draw.rectangle([nx0, ny0, nx1, ny1], outline=(255, 0, 0), width=15)

plt.figure(figsize=(12, 10))
plt.imshow(check_preview)
plt.axis("off")
plt.title(f"{len(negative_boxes_fullres)} negative box(es) mapped back to full-res image")
plt.show()

if neg_crops:
    fig, axes = plt.subplots(1, len(neg_crops), figsize=(4 * len(neg_crops), 4))
    if len(neg_crops) == 1:
        axes = [axes]
    for ax, crop, j in zip(axes, neg_crops, range(len(neg_crops))):
        ax.imshow(crop)
        ax.set_title(f"NEG {j}")
        ax.axis("off")
    plt.show()

In [ ]:
# Visualize GT (yellow) + positives (green) + negatives (red) together
neg_boxes_fullres = negative_boxes_fullres  # alias for clarity below

exemplar_preview = image.copy()
draw = ImageDraw.Draw(exemplar_preview)

for i, (bx0, by0, bx1, by1) in enumerate(gt_boxes):
    draw.rectangle([bx0, by0, bx1, by1], outline=(255, 255, 0), width=20)
    text_x, text_y = int(bx0), max(0, int(by0) - 120)
    draw.rectangle([text_x, text_y, text_x + 120, text_y + 120], fill=(255, 255, 255))
    draw.text((text_x + 10, text_y + 5), str(i), fill=(255, 0, 0), font=font)

HIGHLIGHT_PAD = 15
label_font = ImageFont.truetype(font_path, 90)

for i, box in zip(positive_indices, pos_boxes_fullres):
    px0, py0, px1, py1 = box
    draw.rectangle(
        [px0 - HIGHLIGHT_PAD, py0 - HIGHLIGHT_PAD, px1 + HIGHLIGHT_PAD, py1 + HIGHLIGHT_PAD],
        outline=(0, 255, 0), width=20,
    )
    label_text = f"POS (idx {i})"
    label_x, label_y = int(px0 - HIGHLIGHT_PAD), int(py1 + HIGHLIGHT_PAD + 20)
    text_bbox = draw.textbbox((label_x, label_y), label_text, font=label_font)
    draw.rectangle(text_bbox, fill=(0, 255, 0))
    draw.text((label_x, label_y), label_text, fill=(0, 0, 0), font=label_font)

for j, box in enumerate(neg_boxes_fullres):
    nx0, ny0, nx1, ny1 = box
    draw.rectangle([nx0, ny0, nx1, ny1], outline=(255, 0, 0), width=20)
    label_text = f"NEG {j}"
    label_x, label_y = int(nx0), int(ny1 + 20)
    text_bbox = draw.textbbox((label_x, label_y), label_text, font=label_font)
    draw.rectangle(text_bbox, fill=(255, 0, 0))
    draw.text((label_x, label_y), label_text, fill=(255, 255, 255), font=label_font)

plt.figure(figsize=(12, 10))
plt.imshow(exemplar_preview)
plt.axis("off")
plt.title("GT (yellow) | Positive exemplars (green) | Negative exemplars (red)")
plt.show()

In [ ]:
#  Helper functions (tiling, compositing MULTIPLE exemplars, filtering, NMS)
def get_local_background_patch(tile_img: Image.Image, patch_w: int, patch_h: int) -> Image.Image:
    """
    Sample REAL texture from the tile itself for the strip's background, so color
    balance/brightness/grain match this tile's own lighting and there's no
    artificial white-vs-photo edge for the vision transformer to latch onto.
    """
    sample = tile_img.crop((0, 0, min(patch_w, tile_img.width), min(patch_h, tile_img.height)))
    if sample.size != (patch_w, patch_h):
        sample = sample.resize((patch_w, patch_h))
    return sample

In [ ]:
def make_feather_mask(size, feather_width: int = 8) -> Image.Image:
    """
    Grayscale alpha mask: interior = 255 (opaque), border band ramps to 0, so each
    pasted crop blends smoothly instead of leaving a hard rectangular seam.
    """
    w, h = size
    mask = np.full((h, w), 255, dtype=np.float32)
    for i in range(feather_width):
        alpha = 255.0 * (i + 1) / feather_width
        mask[i, :] = np.minimum(mask[i, :], alpha)
        mask[h - 1 - i, :] = np.minimum(mask[h - 1 - i, :], alpha)
        mask[:, i] = np.minimum(mask[:, i], alpha)
        mask[:, w - 1 - i] = np.minimum(mask[:, w - 1 - i], alpha)
    return Image.fromarray(mask.astype(np.uint8), mode="L")

In [ ]:
def compose_tile_with_exemplars(tile_img: Image.Image, crops: list,
                                 margin: int = 6, feather_width: int = 8):
    """
    Builds the composed image sent to SAM3 for ONE tile: a strip containing ALL
    exemplar crops (positive Rumex crops + negative lookalike crops), laid out
    side-by-side, pasted above the tile content.

    `crops` order must match the `labels` list used later (1 = positive, 0 = negative).

    Returns:
        composed  -- the image to feed to SAM3
        crop_boxes -- list of [x1,y1,x2,y2], one per crop, in composed-image coords
        offset    -- (dx, dy) to convert composed target-region coords back to this
                     TILE's own coordinate system
    """
    strip_h = max(c.height for c in crops) + 2 * margin
    strip_w_needed = sum(c.width for c in crops) + margin * (len(crops) + 1)
    canvas_w = max(tile_img.width, strip_w_needed)
    canvas_h = strip_h + tile_img.height

    strip_bg = get_local_background_patch(tile_img, canvas_w, strip_h)

    composed = Image.new("RGB", (canvas_w, canvas_h))
    composed.paste(strip_bg, (0, 0))
    offset = (0, strip_h)
    composed.paste(tile_img, offset)

    crop_boxes = []
    x_cursor = margin
    for crop in crops:
        feather_mask = make_feather_mask(crop.size, feather_width=feather_width)
        paste_xy = (x_cursor, margin)
        composed.paste(crop, paste_xy, feather_mask)
        crop_boxes.append([
            paste_xy[0], paste_xy[1],
            paste_xy[0] + crop.width, paste_xy[1] + crop.height,
        ])
        x_cursor += crop.width + margin

    return composed, crop_boxes, offset

In [ ]:
def tile_bboxes(img_w: int, img_h: int, tile_size: int, overlap: int):
    """
    Generates (x1,y1,x2,y2) windows covering the whole image, with overlap so plants
    sitting right on a tile boundary aren't missed or half-cut in every window.
    """
    step = tile_size - overlap
    tiles = []
    for y in range(0, img_h, step):
        for x in range(0, img_w, step):
            x2 = min(x + tile_size, img_w)
            y2 = min(y + tile_size, img_h)
            x1 = max(0, x2 - tile_size)
            y1 = max(0, y2 - tile_size)
            tiles.append((x1, y1, x2, y2))
    return list(dict.fromkeys(tiles))

In [ ]:
def keep_only_target_region_detections(boxes, scores, masks, offset, y_tolerance=5):
    """
    Drop any detection sitting inside the padding strip (that's a pasted exemplar
    being re-detected, not a real find), keep only detections within the tile's
    real-content region, and remap coordinates back to the tile's own system.
    """
    dx, dy = offset
    kept_boxes, kept_scores, kept_masks = [], [], []

    for box, score, mask in zip(boxes, scores, masks):
        x1, y1, x2, y2 = box.tolist() if torch.is_tensor(box) else box
        if y1 >= dy - y_tolerance:
            remapped_box = [x1 - dx, max(y1 - dy, 0), x2 - dx, y2 - dy]
            kept_boxes.append(remapped_box)
            kept_scores.append(score)
            mask_np = mask.cpu().numpy() if torch.is_tensor(mask) else mask
            kept_masks.append(mask_np[dy:, dx:] if dx or dy else mask_np)

    return kept_boxes, kept_scores, kept_masks

In [ ]:
def filter_implausible_boxes(boxes, scores, masks, tile_w, tile_h,
                              min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5):
    """
    Drops detections unlikely to be real single Rumex plants: implausibly large
    boxes relative to the tile, weak/noisy low mask-fill detections, or degenerate
    boxes clipped near-zero at tile edges.
    """
    kept_boxes, kept_scores, kept_masks = [], [], []
    tile_area = tile_w * tile_h

    for box, score, mask in zip(boxes, scores, masks):
        x1, y1, x2, y2 = box
        box_w, box_h = x2 - x1, y2 - y1
        if box_w <= edge_margin or box_h <= edge_margin:
            continue

        box_area = box_w * box_h
        if box_area / tile_area > max_area_fraction:
            continue

        mask_np = mask.cpu().numpy() if torch.is_tensor(mask) else mask
        x1c, y1c = int(max(0, x1)), int(max(0, y1))
        x2c, y2c = int(min(mask_np.shape[1], x2)), int(min(mask_np.shape[0], y2))
        if x2c <= x1c or y2c <= y1c:
            continue

        region = mask_np[y1c:y2c, x1c:x2c]
        fill_ratio = (region > 0.5).mean() if region.size > 0 else 0.0
        if fill_ratio < min_fill_ratio:
            continue

        kept_boxes.append(box)
        kept_scores.append(score)
        kept_masks.append(mask)

    return kept_boxes, kept_scores, kept_masks

In [ ]:
def run_sam3_on_tile_multi(composed_tile, crop_boxes, labels, offset, threshold=0.3):
    """
    Runs SAM3 on one composed tile using MULTIPLE exemplar boxes (positive +
    negative, per `labels`) -- no text prompt.
    """
    inputs = sam3_processor(
        images=composed_tile,
        input_boxes=[[[float(c) for c in box] for box in crop_boxes]],
        input_boxes_labels=[labels],
        return_tensors="pt",
    ).to(sam3_model.device)

    with torch.no_grad():
        outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=0.4,
        target_sizes=inputs.get("original_sizes").tolist(),
    )[0]

    return keep_only_target_region_detections(
        results["boxes"], results["scores"], results["masks"], offset
    )

In [ ]:
def nms_merge(boxes, scores, masks, iou_thresh: float = 0.5):
    """
    Because tiles overlap, the same real plant can get detected in two neighboring
    tiles. IoU-based NMS collapses duplicates to the single highest-confidence box,
    carrying the matching mask through too.
    """
    if not boxes:
        return [], [], []

    boxes_t = torch.tensor(boxes, dtype=torch.float32)
    scores_t = torch.tensor([s.item() if torch.is_tensor(s) else s for s in scores])
    order = scores_t.argsort(descending=True)
    keep = []

    while order.numel() > 0:
        i = order[0].item()
        keep.append(i)
        if order.numel() == 1:
            break
        rest = order[1:]
        xx1 = torch.maximum(boxes_t[i, 0], boxes_t[rest, 0])
        yy1 = torch.maximum(boxes_t[i, 1], boxes_t[rest, 1])
        xx2 = torch.minimum(boxes_t[i, 2], boxes_t[rest, 2])
        yy2 = torch.minimum(boxes_t[i, 3], boxes_t[rest, 3])
        inter = (xx2 - xx1).clamp(0) * (yy2 - yy1).clamp(0)
        area_i = (boxes_t[i, 2] - boxes_t[i, 0]) * (boxes_t[i, 3] - boxes_t[i, 1])
        area_r = (boxes_t[rest, 2] - boxes_t[rest, 0]) * (boxes_t[rest, 3] - boxes_t[rest, 1])
        iou = inter / (area_i + area_r - inter + 1e-6)
        order = rest[iou <= iou_thresh]

    kept_boxes = [boxes[i] for i in keep]
    kept_scores = [scores[i] for i in keep]
    kept_masks = [masks[i] for i in keep]
    return kept_boxes, kept_scores, kept_masks

In [ ]:
def show_image_with_boxes(image: Image.Image, boxes_xyxy, masks=None, scores=None,
                           gt_boxes_xyxy=None, title=""):
    """
    masks: list of (mask_np, ox, oy) tuples in TILE-local pixel space, placed at
    (ox, oy) in the full image via matplotlib's `extent`.
    """
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(image)
    ax.set_xlim(0, image.width)
    ax.set_ylim(image.height, 0)

    rng = np.random.default_rng(0)

    if masks is not None:
        for (mask_np, ox, oy) in masks:
            color = rng.uniform(0.2, 1.0, size=3)
            mask_np = mask_np.cpu().numpy() if torch.is_tensor(mask_np) else mask_np
            h, w = mask_np.shape[:2]
            overlay = np.zeros((h, w, 4))
            overlay[mask_np > 0.5] = (*color, 0.45)
            ax.imshow(overlay, extent=[ox, ox + w, oy + h, oy])

    if gt_boxes_xyxy is not None:
        for gx1, gy1, gx2, gy2 in gt_boxes_xyxy:
            rect = patches.Rectangle(
                (gx1, gy1), gx2 - gx1, gy2 - gy1,
                linewidth=2, edgecolor="yellow", facecolor="none",
            )
            ax.add_patch(rect)

    for i, box in enumerate(boxes_xyxy):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect)
        if scores is not None:
            ax.text(
                x1, y1 - 5, f"{scores[i]:.2f}",
                color="lime", fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"),
            )

    ax.set_title(title)
    ax.axis("off")
    plt.show()

In [ ]:
# Preview one composed tile (positive + negative crops, NO text)

TILE_SIZE = 1000
OVERLAP = 150

all_crops = pos_crops + neg_crops
all_labels = [1] * len(pos_crops) + [0] * len(neg_crops)

# Auto-pick the tile containing the first positive exemplar, just for the preview
ex_x0, ex_y0, ex_x1, ex_y1 = pos_boxes_fullres[0]
ex_cx, ex_cy = (ex_x0 + ex_x1) / 2, (ex_y0 + ex_y1) / 2
all_tiles = tile_bboxes(target_image.width, target_image.height, TILE_SIZE, OVERLAP)
preview_x1, preview_y1, preview_x2, preview_y2 = next(
    (t for t in all_tiles if t[0] <= ex_cx < t[2] and t[1] <= ex_cy < t[3]),
    all_tiles[0],
)

preview_tile = target_image.crop((preview_x1, preview_y1, preview_x2, preview_y2))

composed_preview, crop_boxes_preview, offset_preview = compose_tile_with_exemplars(
    preview_tile, all_crops, margin=6, feather_width=8
)

print(f"Tile region in target image: ({preview_x1}, {preview_y1}) -> ({preview_x2}, {preview_y2})")
print(f"Tile size (W, H): {preview_tile.size}")
print(f"Composed image size (W, H): {composed_preview.size}")
print(f"Num exemplar crops in strip: {len(all_crops)} "
      f"({len(pos_crops)} positive, {len(neg_crops)} negative)")
print(f"Crop boxes in composed-image coords: {crop_boxes_preview}")
print(f"Offset (tile content start in composed image): {offset_preview}")

if composed_preview.width != preview_tile.width:
    print(f"NOTE: composed width ({composed_preview.width}) != tile width "
          f"({preview_tile.width}) -- strip is wider than the tile itself.")

show_image_with_boxes(
    composed_preview,
    boxes_xyxy=crop_boxes_preview,
    title=f"Composed tile preview -- {len(pos_crops)} pos + {len(neg_crops)} neg exemplar boxes highlighted",
)

In [ ]:
# Main inference pipeline (positive + negative exemplars, NO text prompt)
"""
Split the large drone image into overlapping tiles.
Composite ALL exemplar crops (positive Rumex crops + negative lookalike crops)
into a strip above each tile. No text prompt is used.
Run SAM3 on each composed tile.
Remove implausible / padding-region detections.
Convert tile-local detections back into original image coordinates.
Merge duplicate detections caused by tile overlap.
"""
all_boxes, all_scores, all_masks = [], [], []

for (x1, y1, x2, y2) in tile_bboxes(target_image.width, target_image.height, TILE_SIZE, OVERLAP):
    tile = target_image.crop((x1, y1, x2, y2))

    composed_tile, crop_boxes, offset = compose_tile_with_exemplars(
        tile, all_crops, margin=6, feather_width=8
    )

    boxes, scores, masks = run_sam3_on_tile_multi(
        composed_tile, crop_boxes, all_labels, offset,
        threshold=0.3,
    )

    tile_w, tile_h = x2 - x1, y2 - y1
    boxes, scores, masks = filter_implausible_boxes(
        boxes, scores, masks, tile_w, tile_h,
        min_fill_ratio=0.15, max_area_fraction=0.6, edge_margin=5,
    )

    for b, s, m in zip(boxes, scores, masks):
        all_boxes.append([b[0] + x1, b[1] + y1, b[2] + x1, b[3] + y1])
        all_scores.append(s.item() if torch.is_tensor(s) else s)
        all_masks.append((m, x1, y1))

final_boxes, final_scores, final_masks = nms_merge(all_boxes, all_scores, all_masks, iou_thresh=0.5)

print(f"Total Rumex instances found: {len(final_boxes)}")
for i, (box, score) in enumerate(zip(final_boxes, final_scores)):
    print(f"  instance {i}: confidence={score:.3f}, box={[round(v, 1) for v in box]}")

In [ ]:
# Display final predictions (lime) vs GT boxes (yellow)

show_image_with_boxes(
    target_image,
    boxes_xyxy=final_boxes,
    masks=final_masks,
    scores=final_scores,
    gt_boxes_xyxy=gt_boxes,
    title=(f"SAM3 (tiled, {len(pos_crops)} pos + {len(neg_crops)} neg exemplars, no text): "
           f"{len(final_boxes)} prediction(s) vs {len(gt_boxes)} GT box(es)"),
)

In [ ]:
# Metrics: mAP50, Precision, Recall, IoU
pred_boxes_np = np.array(final_boxes, dtype=np.float32) if final_boxes else np.zeros((0, 4), dtype=np.float32)
pred_scores_np = np.array(final_scores, dtype=np.float32) if final_scores else np.zeros((0,), dtype=np.float32)

pred_detections = sv.Detections(
    xyxy=pred_boxes_np,
    confidence=pred_scores_np,
    class_id=np.zeros(len(pred_scores_np), dtype=int),
)
gt_detections = sv.Detections(
    xyxy=gt_boxes,
    class_id=np.zeros(len(gt_boxes), dtype=int),
)

from supervision.metrics import MeanAveragePrecision

map_metric = MeanAveragePrecision()
result = map_metric.update([pred_detections], [gt_detections]).compute()
print(f"mAP50: {result.map50:.4f}")

def compute_iou_matrix(boxes1, boxes2):
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)))
    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])
    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])
    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])
    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])
    inter_w = np.clip(x2 - x1, 0, None)
    inter_h = np.clip(y2 - y1, 0, None)
    inter_area = inter_w * inter_h
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union_area = area1[:, None] + area2[None, :] - inter_area
    return np.where(union_area > 0, inter_area / union_area, 0.0)

IOU_THRESHOLD = 0.5
iou_matrix = compute_iou_matrix(pred_boxes_np, gt_boxes)
num_preds, num_gt = len(pred_boxes_np), len(gt_boxes)

matched_gt = set()
true_positives = 0
matched_ious = []
pred_order = np.argsort(-pred_scores_np) if num_preds > 0 else []

for pred_idx in pred_order:
    if num_gt == 0:
        break
    best_gt_idx = np.argmax(iou_matrix[pred_idx])
    best_iou = iou_matrix[pred_idx, best_gt_idx]
    if best_iou >= IOU_THRESHOLD and best_gt_idx not in matched_gt:
        matched_gt.add(best_gt_idx)
        true_positives += 1
        matched_ious.append(best_iou)

false_positives = num_preds - true_positives
false_negatives = num_gt - true_positives
precision = true_positives / num_preds if num_preds > 0 else 0.0
recall = true_positives / num_gt if num_gt > 0 else 0.0
mean_iou_matched = float(np.mean(matched_ious)) if matched_ious else 0.0
mean_iou_all_gt = float(np.sum(matched_ious) / num_gt) if num_gt > 0 else 0.0

print(f"IoU threshold: {IOU_THRESHOLD}")
print(f"Num predictions: {num_preds} | Num GT boxes: {num_gt}")
print(f"True Positives:  {true_positives}")
print(f"False Positives: {false_positives}")
print(f"False Negatives: {false_negatives}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Mean IoU (matched pairs only): {mean_iou_matched:.4f}")
print(f"Mean IoU (over all GT boxes):  {mean_iou_all_gt:.4f}")